[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohar-uzh/thesis/blob/main/notebooks/climatebert_specificity_colab.ipynb)

# ClimateBERT Specificity Classification Pipeline

Classifies **climate-related paragraphs** as **specific** or **non-specific**
using the [ClimateBERT distilroberta-base-climate-specificity](https://huggingface.co/climatebert/distilroberta-base-climate-specificity) model.

**Input:** Detector output Excel file (output of the Climate Detection notebook)
**Output:** Same file with two new columns — `specificity_label` and `specificity_score`

Non-climate paragraphs (`detector_label == 'no'`) are skipped and receive blank values.

**Workflow:** Run Cells 1–4 once per session to set up the environment and load the
model. Then repeat Cell 5 for each input file. Run Cell 6 once at the end to
download all results as a single zip archive.


## 1. Install Dependencies

Installs all required Python packages. Only needs to run once per Colab session.


In [ ]:
!pip install -q transformers accelerate pandas openpyxl tqdm


## 2. Clone / Pull GitHub Repository

Pulls the thesis codebase from GitHub into Colab's filesystem.

- First time: runs `git clone`.
- Returning session: runs `git pull` to fetch any updates.


In [ ]:
import os

if not os.path.exists("/content/thesis"):
    !git clone https://github.com/rohar-uzh/thesis.git /content/thesis
else:
    !cd /content/thesis && git pull

print("✓ Repo ready at /content/thesis")


## 3. Add Repo to Python Path

Tells Python where to find the thesis modules. Must be re-run after every runtime restart.


In [ ]:
import sys
sys.path.insert(0, "/content/thesis")

print("✓ /content/thesis added to Python path")


## 4. Load Model

Loads [climatebert/distilroberta-base-climate-specificity](https://huggingface.co/climatebert/distilroberta-base-climate-specificity) onto the GPU and initialises the classifier.

**Run this cell once per session.** The loaded model is reused for every file
you classify — no reloading between files.


In [ ]:
from climate_specificity import load_classifier, run_specificity_classification

BATCH_SIZE = 64  

clf, id2label = load_classifier()
output_files  = []  # collects output paths for batch download

print(f"  Batch size : {BATCH_SIZE}")
print(f"  Queue size : {len(output_files)} file(s) — ready to classify")


## 5. Upload & Classify

Upload one detector output file and classify it using the already-loaded model.
**Repeat this cell for each input file.** Results are appended to the download queue.

The file must contain a `detector_label` column (i.e. be the output of the Detection notebook).


In [ ]:
from google.colab import files
import shutil, os

# — Upload —
uploaded   = files.upload()
filename   = list(uploaded.keys())[0]

data_dir   = "/content/thesis/data/detected"
os.makedirs(data_dir, exist_ok=True)
input_path = os.path.join(data_dir, filename)
shutil.move(filename, input_path)
print(f"✓ File ready: {input_path}")

# — Classify —
df, output_file = run_specificity_classification(
    input_path=input_path,
    clf=clf,
    id2label=id2label,
    batch_size=BATCH_SIZE,
)
output_files.append(output_file)
print(f"✓ Added to queue: {output_file}  ({len(output_files)} file(s) total)")


## 6. Download All Results

Compresses all classified files (each with `specificity_label` (spec/not-spec) and `specificity_score` (confidence 0–1)) into a single zip archive and downloads it.

**Run once after processing all your files.**


In [ ]:
import zipfile
from google.colab import files

if not output_files:
    print("⚠  No files to download yet. Run the Upload & Classify cell first.")
else:
    zip_path = "/content/results_specificity.zip"
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in output_files:
            zf.write(f, os.path.basename(f))
    files.download(zip_path)
    print(f"✓ Downloaded {len(output_files)} file(s) as results_specificity.zip")
